# Real Estate Intelligence Newsletter Agent

## Overview
A **multi-stage agentic AI system** using the OpenAI Agents SDK. For Real estate companies, Investors, Market analysts and Property platforms.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
# Trace provider reads this at `import agents` time.
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "true"

from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from agents.tracing import default_exporter, set_tracing_disabled

set_tracing_disabled(True)
# Ingest uses OPENAI_API_KEY; OpenRouter keys are invalid for api.openai.com — skip HTTP export.
default_exporter().set_api_key("")

from pydantic import BaseModel
from openai import AsyncOpenAI
from typing import List, Optional, Literal, Tuple
import gradio as gr
import asyncio
import httpx

API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = "openai/gpt-4o-mini"

In [ ]:
client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
)
openrouter_model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)

In [ ]:
@function_tool
def web_search(query: str) -> str:
    """Search the public web for current real estate and market information. Pass a focused search query."""
    try:
        r = httpx.get(
            "https://api.duckduckgo.com/",
            params={"q": query, "format": "json", "no_html": 1},
            timeout=30.0,
            headers={
                "User-Agent": "Mozilla/5.0 (compatible; research-agent/1.0; +https://openrouter.ai)"
            },
        )
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        return f"Search request failed: {e}"

    parts: List[str] = []
    if data.get("Heading"):
        parts.append(f"## {data['Heading']}")
    if data.get("Abstract"):
        parts.append(data["Abstract"])
    if data.get("AbstractURL"):
        parts.append(f"Source: {data['AbstractURL']}")
    for t in data.get("RelatedTopics", [])[:10]:
        if isinstance(t, dict) and t.get("Text"):
            parts.append(t["Text"])
        elif isinstance(t, dict) and "Topics" in t:
            for sub in t.get("Topics", [])[:2]:
                if isinstance(sub, dict) and sub.get("Text"):
                    parts.append(sub["Text"])
    if not parts:
        return (
            "No web snippets returned for this query. Answer from general knowledge where appropriate."
        )
    return "\n\n".join(parts)

In [ ]:
class IntakeResult(BaseModel):
    status: Literal["ok", "blocked"]
    reason: Optional[str] = None
    topic: Optional[str] = None
    audience: Optional[str] = None
    region: Optional[str] = None


INTAKE_INSTRUCTIONS = """
You are an intake guardrail for a real estate intelligence system.

Allow ONLY:
- real estate
- housing
- property investment
- rental market
- real estate trends

Block:
- unrelated topics (health, politics, random content)

If valid:
- extract topic, audience, region

If invalid:
- return blocked with reason
"""


intake_agent = Agent(
    name="RealEstateGuardrail",
    instructions=INTAKE_INSTRUCTIONS,
    model=openrouter_model,
    output_type=IntakeResult,
)


async def run_intake(user_input: str) -> IntakeResult:
    result = await Runner.run(intake_agent, user_input)
    return result.final_output

In [ ]:
class PlannerOutput(BaseModel):
    searches: List[str]


planner_agent = Agent(
    name="PlannerAgent",
    instructions=(
        "Generate 2–3 search queries for real estate market research.\n"
        "Focus on trends, pricing, and investment insights."
    ),
    model=openrouter_model,
    output_type=PlannerOutput,
)


async def run_planner(intake: IntakeResult):
    result = await Runner.run(
        planner_agent,
        f"{intake.topic} in {intake.region} for {intake.audience}"
    )
    return result.final_output

In [ ]:
search_agent = Agent(
    name="ResearchAgent",
    instructions=(
        "Research real estate trends and summarize key insights. "
        "You MUST call the web_search tool at least once with a focused query, then synthesize findings."
    ),
    tools=[web_search],
    model=openrouter_model,
)


async def run_search(query: str):
    result = await Runner.run(search_agent, query)
    return str(result.final_output)


async def run_all_searches(queries: List[str]):
    results = []
    for q in queries:
        summary = await run_search(q)
        results.append({"query": q, "summary": summary})
    return results

In [ ]:
analysis_agent = Agent(
    name="AnalysisAgent",
    instructions=(
        "Analyze research and extract:\n"
        "- key trends\n"
        "- opportunities\n"
        "- risks"
    ),
    model=openrouter_model,
)


async def run_analysis(research_data):
    result = await Runner.run(analysis_agent, str(research_data))
    return str(result.final_output)

In [ ]:
@function_tool
def mailersend_email(
    to: str,
    subject: str,
    body_markdown: str,
) -> str:
    """
    Send email using MailerSend API.
    Returns: success or failure message.
    """
    if not to:
        return "❌ No recipient provided"

    mg_key = os.getenv("MAILERSEND_API_KEY")
    domain_id = os.getenv("MAILERSEND_DOMAIN_ID")
    from_email = os.getenv("EMAIL_FROM", f"newsletter@{domain_id}")

    import markdown
    html_body = markdown.markdown(body_markdown)

    payload = {
        "from": {"email": from_email, "name": "Real Estate Newsletter"},
        "to": [{"email": to}],
        "subject": subject,
        "text": body_markdown,
        "html": html_body,
    }

    headers = {"Authorization": f"Bearer {mg_key}"}

    try:
        r = httpx.post(
            "https://api.mailersend.com/v1/email",
            headers=headers,
            json=payload,
            timeout=30,
        )
        r.raise_for_status()
    except Exception as e:
        return f"❌ MailerSend failed: {e}"

    return f"📧 Newsletter successfully sent to {to}"

In [ ]:
writer_instructions = """
You are a real estate newsletter writer.

1. Write a Markdown newsletter.
2. Include a subject line.
3. If the user included an email address, call mailersend_email(to, subject, body_markdown).
"""

writer_agent = Agent(
    name="RealEstateWriter",
    instructions=writer_instructions,
    tools=[mailersend_email],
    model=openrouter_model,
)

In [ ]:
def to_evidence_md(bundle: list) -> str:
    lines = ["## Research snippets", ""]
    for item in bundle:
        lines.append(f"### Query: {item['query']}\n{item['summary']}\n")
    return "\n".join(lines)


async def run_full_pipeline(user_text: str, marketing_email: str) -> str:
    # Guardrail
    intake = await run_intake(user_text)
    if intake.status != "ok":
        return f"❌ Intake blocked: {intake.reason}"

    # Planner
    plan = await run_planner(intake)

    # Search (pass list of query strings, not the PlannerOutput model)
    bundle = await run_all_searches(plan.searches)

    # Analysis
    analysis_text = await run_analysis(bundle)

    # Compose evidence + analysis
    evidence_md = to_evidence_md(bundle)
    full_md = f"{evidence_md}\n\n{analysis_text}"

    # Writer + Email
    prompt = (
        f"Topic: {intake.topic}\n"
        f"Region: {intake.region}\n"
        f"Audience: {intake.audience}\n\n"
        f"{full_md}\n\n"
        f"Marketing email: {marketing_email}"
    )

    res = await Runner.run(writer_agent, prompt)
    return str(res.final_output)

In [ ]:
def gradio_run(topic, region, audience, marketing_email):
    topic = (topic or "").strip()
    region = (region or "").strip()
    audience = (audience or "").strip()
    marketing_email = (marketing_email or "").strip()

    if not topic:
        return "Please enter a real estate topic."

    user_input = (
        f"Newsletter request: {topic}, region: {region}, audience: {audience}"
    )

    return asyncio.run(run_full_pipeline(user_input, marketing_email))


demo = gr.Interface(
    fn=gradio_run,
    inputs=[
        gr.Textbox(label="Real Estate Topic", lines=2),
        gr.Textbox(label="Region / Market"),
        gr.Textbox(label="Audience"),
        gr.Textbox(label="Marketing Email to send"),
    ],
    outputs=gr.Markdown(label="Newsletter + Email Status"),
    title="Real Estate Newsletter + MailerSend Email",
    description="Generates and optionally emails a real estate marketing newsletter.",
)

demo.launch()

# Future Improvements

### 1. Add Location Intelligence
- Integrate geo-based property APIs
- Add price heatmaps

### 2. Add Personalization
- User-specific investor preferences

### 3. Add Ranking Engine
- Combine deterministic + LLM scoring

### 4. Add Persistence
- Store previous newsletters
- Trend tracking over time

### 5. Email
- Integrate with email templates and email addresses dynamically.